# Step Independence Analysis

Ask an LLM judge whether each consecutive pair of steps in a ReAct trajectory has Step B re-attempting the same action Step A already attempted (e.g. a reworded repeat search, re-visiting the same page, retrying an equivalent approach), as opposed to Step B building on Step A's result to take a genuinely new action -- even one on the same topic or source. Consecutive "yes" verdicts form a *run*; we sum `token_usage.total_tokens` over the whole run to get the total cost of that repeated-action streak, not just an isolated pair.

In [1]:
import os
import pickle
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
judge_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

JUDGE_MODEL = "gpt-4o-mini"

In [2]:
JUDGE_SYSTEM_PROMPT = """You are evaluating one AI agent's step-by-step trajectory while solving a task.
You will be shown the task, then two consecutive steps (each a "thought" followed by a code snippet) taken by the agent.
Decide whether Step B is RE-ATTEMPTING the same action Step A already attempted -- e.g. repeating a similar search
query, re-visiting the same page, re-extracting the same piece of information, retrying an equivalent approach --
as opposed to Step B using or building on what Step A did/found to take a NEW action, even if that new action
concerns the same topic, source, or document as Step A.
Answer with exactly one word: "yes" or "no"."""


def format_step(step: dict) -> str:
    return step.get("model_output") or ""


def same_subgoal(question: str, step_a: dict, step_b: dict, model: str = JUDGE_MODEL) -> bool:
    """Ask the judge model whether step_b is re-attempting the same action as step_a."""
    user_content = (
        f"Task: {question}\n\n"
        f"=== Step A ===\n{format_step(step_a)}\n\n"
        f"=== Step B ===\n{format_step(step_b)}"
    )
    response = judge_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        temperature=0,
    )
    verdict = response.choices[0].message.content.strip().lower()
    return verdict.startswith("yes")

In [3]:
import json
import re
from concurrent.futures import ThreadPoolExecutor

KNOWN_TOOLS = [
    "web_search", "visit_page", "page_up", "page_down", "find_on_page_ctrl_f",
    "find_next", "find_archived_url", "inspect_file_as_text", "visualizer", "final_answer",
]


def tools_in_step(step: dict) -> set[str]:
    code = step.get("code_action") or ""
    return {t for t in KNOWN_TOOLS if re.search(rf"\b{re.escape(t)}\s*\(", code)}


def step_details(step: dict) -> dict:
    return {
        "step_number": step.get("step_number"),
        "model_output": step.get("model_output"),
        "code_action": step.get("code_action"),
        "observations": step.get("observations"),
        "token_usage": step.get("token_usage"),
    }


def scan_trajectory(traj: dict, executor: ThreadPoolExecutor) -> list[dict]:
    """Run the judge over every consecutive pair in one trajectory (concurrently); return only the
    redundant ones, with full step details included so findings can be written out and inspected later."""
    steps = traj["steps"]
    pairs = list(range(len(steps) - 1))
    verdicts = executor.map(lambda i: same_subgoal(traj["question"], steps[i], steps[i + 1]), pairs)

    findings = []
    for i, verdict in zip(pairs, verdicts):
        if verdict:
            findings.append({
                "task_id": traj.get("task_id", "?"),
                "question": traj["question"],
                "step_i": i,
                "step_j": i + 1,
                "tools": sorted(tools_in_step(steps[i]) | tools_in_step(steps[i + 1])),
                "steps": {
                    str(i): step_details(steps[i]),
                    str(i + 1): step_details(steps[i + 1]),
                },
            })
    return findings

In [4]:
MARKOV_DIR = Path("../markovReAct")

MODEL_DIRS = {
    # naive baselines (already scanned -- kept under their original keys so the cache hits)
    "gpt-4o": Path("../baseline/naive_react_gpt-4o_False"),
    "gpt-5.4-mini": Path("../baseline/naive_react_gpt-5.4-mini_False"),
    "Qwen3.7-Plus": Path("../baseline/naive_react_Qwen/Qwen3.7-Plus_False"),
    "Qwen3.5-9B": Path("../baseline/naive_react_Qwen/Qwen3.5-9B_False"),
    # markovReAct, window_size in {1, 3, 5}
    "gpt-4o_w1": MARKOV_DIR / "markov_react_gpt-4o_w1",
    "gpt-4o_w3": MARKOV_DIR / "markov_react_gpt-4o_w3",
    "gpt-4o_w5": MARKOV_DIR / "markov_react_gpt-4o_w5",
    "gpt-5.4-mini_w1": MARKOV_DIR / "markov_react_gpt-5.4-mini_w1",
    "gpt-5.4-mini_w3": MARKOV_DIR / "markov_react_gpt-5.4-mini_w3",
    "gpt-5.4-mini_w5": MARKOV_DIR / "markov_react_gpt-5.4-mini_w5",
    "Qwen3.7-Plus_w1": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.7-Plus_w1",
    "Qwen3.7-Plus_w3": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.7-Plus_w3",
    "Qwen3.7-Plus_w5": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.7-Plus_w5",
    "Qwen3.5-9B_w1": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.5-9B_w1",
    "Qwen3.5-9B_w3": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.5-9B_w3",
    "Qwen3.5-9B_w5": MARKOV_DIR / "markov_react_Qwen" / "Qwen3.5-9B_w5",
}


def load_cache(output_path: Path) -> tuple[set[str], list[dict]]:
    """Load {scanned_task_ids, findings} from output_path if present."""
    if not output_path.exists():
        return set(), []
    with open(output_path) as f:
        data = json.load(f)
    return set(data.get("scanned_task_ids", [])), data.get("findings", [])


results_by_model = {}
with ThreadPoolExecutor(max_workers=10) as executor:
    for model_name, pkl_dir in MODEL_DIRS.items():
        output_path = Path(f"redundant_pairs_{model_name}.json")
        scanned_task_ids, all_findings = load_cache(output_path)
        print(scanned_task_ids)
        if scanned_task_ids:
            print(f"{model_name}: resuming, {len(scanned_task_ids)} trajectory(ies) already cached")

        for pkl_file in sorted(pkl_dir.glob("*.pkl"), key=lambda p: int(p.stem)):
            with open(pkl_file, "rb") as f:
                traj = pickle.load(f)
            if "steps" not in traj:
                continue
            task_id = traj.get("task_id", "?")
            if task_id in scanned_task_ids:
                continue

            all_findings.extend(scan_trajectory(traj, executor))
            scanned_task_ids.add(task_id)

            # Persist after every trajectory so an interrupted run only loses in-flight work,
            # not everything scanned so far.
            with open(output_path, "w") as f:
                json.dump(
                    {"scanned_task_ids": sorted(scanned_task_ids), "findings": all_findings},
                    f, indent=2, default=str,
                )

        results_by_model[model_name] = all_findings
        print(f"{model_name}: {len(all_findings)} redundant pair(s) across {len(scanned_task_ids)} trajectories -> {output_path}")

{'872bfbb1-9ccf-49f6-8c5f-aa22818ccd66', '9b54f9d9-35ee-4a14-b62f-d130ea00317f', 'e2d69698-bc99-4e85-9880-67eaccd66e6c', '08c0b6e9-1b43-4c2e-ae55-4e3fce2c2715', 'c3a79cfe-8206-451f-aca8-3fec8ebe51d3', '8e867cd7-cff9-4e6c-867a-ff5ddc2550be', 'ad2b4d70-9314-4fe6-bfbe-894a45f6055f', 'd0633230-7067-47a9-9dbf-ee11e0a2cdd6', '73c1b9fe-ee1d-4cf4-96ca-35c08f97b054', '851e570a-e3de-4d84-bcfa-cc85578baa59', '71345b0a-9c7d-4b50-b2bf-937ec5879845', '08cae58d-4084-4616-b6dd-dd6534e4825b', '114d5fd0-e2ae-4b6d-a65a-870da2d19c08', '853c8244-429e-46ca-89f2-addf40dfb2bd', '8131e2c0-0083-4265-9ce7-78c2d568425d', 'c8b7e059-c60d-472e-ad64-3b04ae1166dc', 'dd3c7503-f62a-4bd0-9f67-1b63b94194cc', '16d825ff-1623-4176-a5b5-42e0f5c2b0ac', 'dc28cf18-6431-458b-83ef-64b3ce566c10', 'e4e91f1c-1dcd-439e-9fdd-cb976f5293fd', '14569e28-c88c-43e4-8c32-097d35b9a67d', 'e1fc63a2-da7a-432f-be78-7c4a95598703', 'c526d8d6-5987-4da9-b24c-83466fa172f3', 'a0c07678-e491-4bbc-8f0b-07405144218f', '6359a0b1-8f7b-499b-9336-840f9ab90688',

gpt-4o: 381 redundant pair(s) across 165 trajectories -> redundant_pairs_gpt-4o.json
{'872bfbb1-9ccf-49f6-8c5f-aa22818ccd66', '9b54f9d9-35ee-4a14-b62f-d130ea00317f', 'e2d69698-bc99-4e85-9880-67eaccd66e6c', '08c0b6e9-1b43-4c2e-ae55-4e3fce2c2715', 'c3a79cfe-8206-451f-aca8-3fec8ebe51d3', '8e867cd7-cff9-4e6c-867a-ff5ddc2550be', 'ad2b4d70-9314-4fe6-bfbe-894a45f6055f', 'd0633230-7067-47a9-9dbf-ee11e0a2cdd6', '73c1b9fe-ee1d-4cf4-96ca-35c08f97b054', '851e570a-e3de-4d84-bcfa-cc85578baa59', '71345b0a-9c7d-4b50-b2bf-937ec5879845', '08cae58d-4084-4616-b6dd-dd6534e4825b', '114d5fd0-e2ae-4b6d-a65a-870da2d19c08', '853c8244-429e-46ca-89f2-addf40dfb2bd', '8131e2c0-0083-4265-9ce7-78c2d568425d', 'c8b7e059-c60d-472e-ad64-3b04ae1166dc', 'dd3c7503-f62a-4bd0-9f67-1b63b94194cc', '16d825ff-1623-4176-a5b5-42e0f5c2b0ac', 'dc28cf18-6431-458b-83ef-64b3ce566c10', 'e4e91f1c-1dcd-439e-9fdd-cb976f5293fd', '14569e28-c88c-43e4-8c32-097d35b9a67d', 'e1fc63a2-da7a-432f-be78-7c4a95598703', 'c526d8d6-5987-4da9-b24c-83466fa17

Qwen3.7-Plus: 710 redundant pair(s) across 165 trajectories -> redundant_pairs_Qwen3.7-Plus.json
{'872bfbb1-9ccf-49f6-8c5f-aa22818ccd66', '9b54f9d9-35ee-4a14-b62f-d130ea00317f', 'e2d69698-bc99-4e85-9880-67eaccd66e6c', '08c0b6e9-1b43-4c2e-ae55-4e3fce2c2715', 'c3a79cfe-8206-451f-aca8-3fec8ebe51d3', '8e867cd7-cff9-4e6c-867a-ff5ddc2550be', 'ad2b4d70-9314-4fe6-bfbe-894a45f6055f', 'd0633230-7067-47a9-9dbf-ee11e0a2cdd6', '73c1b9fe-ee1d-4cf4-96ca-35c08f97b054', '851e570a-e3de-4d84-bcfa-cc85578baa59', '71345b0a-9c7d-4b50-b2bf-937ec5879845', '08cae58d-4084-4616-b6dd-dd6534e4825b', '114d5fd0-e2ae-4b6d-a65a-870da2d19c08', '853c8244-429e-46ca-89f2-addf40dfb2bd', '8131e2c0-0083-4265-9ce7-78c2d568425d', 'c8b7e059-c60d-472e-ad64-3b04ae1166dc', 'dd3c7503-f62a-4bd0-9f67-1b63b94194cc', '16d825ff-1623-4176-a5b5-42e0f5c2b0ac', 'dc28cf18-6431-458b-83ef-64b3ce566c10', 'e4e91f1c-1dcd-439e-9fdd-cb976f5293fd', '14569e28-c88c-43e4-8c32-097d35b9a67d', 'e1fc63a2-da7a-432f-be78-7c4a95598703', 'c526d8d6-5987-4da9-b2

Qwen3.5-9B: 632 redundant pair(s) across 165 trajectories -> redundant_pairs_Qwen3.5-9B.json
set()


gpt-4o_w1: 796 redundant pair(s) across 165 trajectories -> redundant_pairs_gpt-4o_w1.json
set()


...

Qwen3.7-Plus_w3: 1124 redundant pair(s) across 165 trajectories -> redundant_pairs_Qwen3.7-Plus_w3.json
set()


Qwen3.7-Plus_w5: 950 redundant pair(s) across 165 trajectories -> redundant_pairs_Qwen3.7-Plus_w5.json
set()


Qwen3.5-9B_w1: 1350 redundant pair(s) across 165 trajectories -> redundant_pairs_Qwen3.5-9B_w1.json
set()


Qwen3.5-9B_w3: 1096 redundant pair(s) across 165 trajectories -> redundant_pairs_Qwen3.5-9B_w3.json
set()


Qwen3.5-9B_w5: 908 redundant pair(s) across 165 trajectories -> redundant_pairs_Qwen3.5-9B_w5.json


In [ ]:
def total_pairs(pkl_dir: Path) -> int:
    """Total number of consecutive step-pairs across every trajectory in pkl_dir."""
    total = 0
    for pkl_file in pkl_dir.glob("*.pkl"):
        with open(pkl_file, "rb") as f:
            traj = pickle.load(f)
        total += max(0, traj.get("num_steps", 0) - 1)
    return total


rows = []
for model_name, pkl_dir in MODEL_DIRS.items():
    output_path = Path(f"redundant_pairs_{model_name}.json")
    with open(output_path) as f:
        data = json.load(f)
    n_redundant = len(data["findings"])
    n_total = total_pairs(pkl_dir)
    rows.append({
        "config": model_name,
        "redundant_pairs": n_redundant,
        "total_pairs": n_total,
        "fraction_redundant": n_redundant / n_total if n_total else None,
    })

norm_df = pd.DataFrame(rows)
norm_df["fraction_redundant_pct"] = norm_df["fraction_redundant"].map(lambda x: f"{x:.1%}" if x is not None else "-")
norm_df